<a href="https://colab.research.google.com/github/madhulikajaibalaji-debug/AGILE/blob/main/Final_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gdown

In [2]:
import gdown

file_id = "1Bd9kyNEt31sk3y-hvg_iDTh9zxNY2xrL"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "test.csv", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1Bd9kyNEt31sk3y-hvg_iDTh9zxNY2xrL
From (redirected): https://drive.google.com/uc?id=1Bd9kyNEt31sk3y-hvg_iDTh9zxNY2xrL&confirm=t&uuid=541fd032-3649-482f-9d59-8a0573cffd0a
To: /content/test.csv
100%|██████████| 1.03G/1.03G [00:17<00:00, 58.1MB/s]


'test.csv'

In [7]:
# 1. Install the required PyDrive library (this only takes a few seconds)
!pip install pydrive

In [8]:
# 1. Authenticate the user (your team will click 'Allow' here)
from google.colab import auth
auth.authenticate_user()

# 2. Import Google's official built-in API tools
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io

# 3. Connect to Google Drive securely as the logged-in user
drive_service = build('drive', 'v3')

# 4. Set your file details
file_id = '1DYvQFLmtB1303tw2jYK5r3xGn4-ViN9z'
file_name = 'train.csv'

print(f"Requesting {file_name} from Drive...")
request = drive_service.files().get_media(fileId=file_id)

# 5. Download the file and show a progress bar
with open(file_name, 'wb') as f:
    downloader = MediaIoBaseDownload(f, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        if status:
            print(f"Download progress: {int(status.progress() * 100)}%")

print("Download complete! Your dataset is ready to use.")

Requesting train.csv from Drive...


Download progress: 1%
Download progress: 2%
Download progress: 3%
Download progress: 4%
Download progress: 5%
Download progress: 7%
Download progress: 8%
Download progress: 9%
Download progress: 10%
Download progress: 11%
Download progress: 13%
Download progress: 14%
Download progress: 15%
Download progress: 16%
Download progress: 17%
Download progress: 19%
Download progress: 20%
Download progress: 21%
Download progress: 22%
Download progress: 23%
Download progress: 25%
Download progress: 26%
Download progress: 27%
Download progress: 28%
Download progress: 29%
Download progress: 31%
Download progress: 32%
Download progress: 33%
Download progress: 34%
Download progress: 35%
Download progress: 37%
Download progress: 38%
Download progress: 39%
Download progress: 40%
Download progress: 41%
Download progress: 43%
Download progress: 44%
Download progress: 45%
Download progress: 46%
Download progress: 47%
Download progress: 49%
Download progress: 50%
Download progress: 51%
Download progress: 

In [10]:
import pandas as pd
import re
import hashlib
from sklearn.utils import resample

def clean_code(code):
    """Deep cleans C/C++ code to focus on logic, not formatting."""
    if not isinstance(code, str): return ""
    # 1. Remove comments (// and /* */)
    code = re.sub(r'//.*', '', code)
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    # 2. Normalize whitespace (removes tabs/newlines)
    code = re.sub(r'\s+', ' ', code).strip()
    return code

def process_and_balance(file_path, output_name, balance=False):
    """
    Cleans, deduplicates, and optionally balances the dataset.
    """
    df = pd.read_csv(file_path)
    code_col = 'func_before' if 'func_before' in df.columns else 'code'

    # 1. CLEANING & DEDUPLICATION
    df[code_col] = df[code_col].apply(clean_code)
    # Use MD5 to find and drop duplicates (prevents data leakage)
    df['hash'] = df[code_col].apply(lambda x: hashlib.md5(x.encode()).hexdigest())
    df = df.drop_duplicates(subset=['hash']).drop(columns=['hash'])

    # 2. BALANCING (Only for Train samples)
    if balance:
        df_vul = df[df['target'] == 1]
        df_clean = df[df['target'] == 0]

        # Downsample the majority 'Clean' class to match 'Vulnerable' class
        df_clean_balanced = resample(df_clean,
                                     replace=False,
                                     n_samples=len(df_vul),
                                     random_state=42)
        df = pd.concat([df_vul, df_clean_balanced])
        print(f"Balanced {file_path}: 50/50 split ({len(df_vul)} each).")
    else:
        print(f"Cleaned {file_path}: Kept original distribution.")

    # Shuffle and Save
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    df.to_csv(output_name, index=False)

# --- EXECUTION ---
# Train: BALANCED (to help models learn)
process_and_balance('train.csv', 'train_final.csv', balance=True)

# Test & Val: ORIGINAL RATIO (to reflect real world)
process_and_balance('test.csv', 'test_final.csv', balance=False)

Balanced train.csv: 50/50 split (8279 each).
Cleaned test.csv: Kept original distribution.
